<div style="padding:20px;color:white;margin:0;font-size:175%;text-align:center;display:fill;border-radius:5px;background-color:#90EE90;overflow:hidden;font-weight:500;color:black">Titanic Ensemble</div>

<img src = "https://cdn.pixabay.com/photo/2015/02/01/10/18/ensemble-619258_960_720.jpg" width = 900 height = 900/>

Ensembling is a technique that combine predictions from base models to improve performance. The top scores in Kaggle competitions are often Ensemble of base models.
Here are our base models :
* [XGBoost](https://www.kaggle.com/code/gehallak/titanic-xgb)
* [Pytorch Neural Network](https://www.kaggle.com/code/gehallak/titanic-pytorch-nn-columntransformer)
* [CatBoost](https://www.kaggle.com/code/gehallak/titanic-catboost)


We will combine them and see if the result is better than the best of them.

<div style="padding:20px;color:white;margin:0;font-size:175%;text-align:center;display:fill;border-radius:5px;background-color:#90EE90;overflow:hidden;font-weight:500;color:black">Merge the base models</div>

We will merge in a single DataFrame (**Ensemble**) the base models submission DataFrames

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
models=[
        'xgboost',
        'pytorch-NN',
        'catboost'
       ]

sub_dfs=['../input/titanic-xgb/submission.csv',
        '../input/titanic-pytorch-nn-columntransformer/submission.csv',
         '../input/titanic-catboost/submission.csv',
        ]

In [ ]:
ensStarted=False
Ensemble=None
for model,sub_df in zip(models,sub_dfs):
    sub = pd.read_csv(sub_df)
    sub.rename(columns={"Survived": model},inplace=True)
    if ensStarted : 
        Ensemble = pd.merge(Ensemble, sub, how='inner', on='PassengerId')  
    else :
        Ensemble=sub
        ensStarted=True 
Ensemble    

<div style="padding:20px;color:white;margin:0;font-size:175%;text-align:center;display:fill;border-radius:5px;background-color:#90EE90;overflow:hidden;font-weight:500;color:black">L0 Predictions</div>

We call Level 0 or L0 the base models and Level 1 or L1 the combining model.

Let's compare the L0 Survival Rates

In [ ]:
matplotlib.rcParams.update({'font.size': 14})
fig, ax = plt.subplots(ncols=len(models),figsize=(20,5))
i=0
for model in models:
    sizes=[Ensemble[model].sum(),len(Ensemble)-Ensemble[model].sum()]
    labels=['Survived','Perished']
    ax[i].pie(sizes, labels=labels, autopct='%1.0f%%',
        shadow=True, startangle=90)
    ax[i].axis('equal') 
    ax[i].set_title(model)
    i+=1
plt.show()

The base models have approximately the same survival rate.

Now let's see how often the base models **agree** on their prediction. 

In [ ]:
Ensemble['Agree']=Ensemble[models].eq(Ensemble[models[0]],axis=0).all(1)

matplotlib.rcParams.update({'font.size': 15})
fig, ax = plt.subplots(ncols=1,figsize=(15,10))
sizes=[Ensemble['Agree'].sum(),len(Ensemble)-Ensemble['Agree'].sum()]
labels=['Agree','Disagree']
ax.pie(sizes, labels=labels, autopct='%1.0f%%',
        shadow=True, startangle=0)
ax.axis('equal') 
ax.set_title('L0 models consensus')
plt.show()

The models **disagree on 12%** of their prediction. 

Next, let's compare the L0 models accuracy.

In [ ]:
scores=[#0.77511,#Light GBM
        0.77751,#XGBoost
        0.76794,#pytorch-NN
        0.78468,#CatBoost
       ]
matplotlib.rcParams.update({'font.size': 14})
fig, ax = plt.subplots(ncols=len(models),figsize=(20,5))
i=0
for model,score in zip(models,scores):
    sizes=[score,1-score]
    labels=['correct','wrong']
    ax[i].pie(sizes, labels=labels, autopct='%1.2f%%',
        shadow=True, startangle=90)
    ax[i].axis('equal') 
    ax[i].set_title(model)
    i+=1
plt.show()

CatBoost has the best score and Pytorch-NN the worst.

We want the L0 models to be wrong and right for different passengers. They will correct each other and the L1 model will be better than the best of the L0 models. One way to look for this is the correlation matrix. 

In [ ]:
corr = Ensemble[models].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, 
        xticklabels=corr.columns,
        yticklabels=corr.columns)
plt.title('L0 models correlation')
plt.show()

Lowly correlated models generally ensemble well. Here XGBoost and pytorch-NN have a low correlation and we can expect the L1 predictions to have a higher score than the max of L0 scores. We keep pytorch-NN, even if its score is relatively lower, because it is lowly correlated. In general, Neural Network models see things in a different way than Trees models and ensemble well with them.

<div style="padding:20px;color:white;margin:0;font-size:175%;text-align:center;display:fill;border-radius:5px;background-color:#90EE90;overflow:hidden;font-weight:500;color:black">Combine L0 predictions and submit</div>

We will simply use the average of the L0 predictions to calculate the L1 prediction.

In [ ]:
Ensemble['Survived']=np.round(Ensemble.loc[:,models].sum(axis=1)/len(models)).astype(int)
Ensemble

In [ ]:
Ensemble[['PassengerId',"Survived"]].to_csv('submission.csv',index=False)

<div style="padding:20px;color:white;margin:0;font-size:175%;text-align:center;display:fill;border-radius:5px;background-color:#90EE90;overflow:hidden;font-weight:500;color:black">Conclusion</div>

the ensemble score is **0.78947**, which is an improvement to **0.78468**, the best of the L0 model scores.